# 2. Internal Data Model and Schema Layer

## Schema Objective

Once raw external records were retrieved, the next requirement was to convert them into a stable internal structure that downstream logic could use consistently.

The purpose of this layer was not to preserve raw source complexity. The purpose was to define a reusable internal data model that could separate source-specific retrieval details from later analytical computation.

Without such a schema layer, each downstream function would need to repeatedly interpret raw source fields, handle missing values independently, and reconstruct the same assumptions every time a new metric or output view was added. That would make the workflow harder to extend, harder to debug, and less stable as the project grew.

For that reason, the schema layer was introduced as the internal analytical boundary between retrieval and metric engineering.

## Internal Data Model

The core internal unit of the project was the annual `FinancialSnapshot`.

Each snapshot represented one fiscal-year record for one company and served as the standard internal data contract for the analytical workflow. Instead of letting downstream logic depend directly on source-specific responses, the pipeline first reorganized retrieved values into this normalized annual structure.

This design had two advantages.

First, it matched the structure of the later analysis, which depended heavily on year-based comparison across revenue, income, cash flow, financial condition, and valuation context.

Second, it created a clean boundary between ingestion logic and analytical logic. Once the annual records were constructed, downstream functions no longer needed to know how each field had originally been retrieved.

In [ ]:
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any

@dataclass
class FinancialSnapshot:
    fiscal_year: int

    # Income statement
    revenue: Optional[float] = None
    net_income: Optional[float] = None
    operating_income: Optional[float] = None

    # Cash flow statement
    operating_cash_flow: Optional[float] = None
    capex: Optional[float] = None

    # Balance sheet
    cash_and_equivalents: Optional[float] = None
    total_debt: Optional[float] = None
    total_equity: Optional[float] = None
    total_assets: Optional[float] = None
    total_liabilities: Optional[float] = None
    current_assets: Optional[float] = None
    current_liabilities: Optional[float] = None

    # Per-share data
    shares_outstanding_basic: Optional[float] = None
    shares_outstanding_diluted: Optional[float] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

## Schema Population Logic

The schema layer was not limited to defining a class structure. It also required a consistent mapping process that translated raw source values into the internal annual record.

This population step determined which source fields were retained, how they were mapped into normalized variables, and which subset of the raw records became part of the reusable analytical dataset.

The field selection was intentional. Revenue, income, and operating cash flow supported growth and profitability analysis. Capital expenditure enabled free-cash-flow construction. Debt, equity, cash, and current-account fields supported financial condition metrics. Share-count fields supported per-share and valuation-related outputs.

In that sense, schema construction was not only a formatting step. It was also the stage at which the future scope of the analytical workflow was determined.

In [ ]:
snapshot = FinancialSnapshot(
    fiscal_year=year,
    revenue=_extract_latest_value_for_year(
        facts,
        tags=["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet"],
        fiscal_year=year,
    ),
    net_income=_extract_latest_value_for_year(
        facts,
        tags=["NetIncomeLoss"],
        fiscal_year=year,
    ),
    operating_income=_extract_latest_value_for_year(
        facts,
        tags=["OperatingIncomeLoss"],
        fiscal_year=year,
    ),
    operating_cash_flow=_extract_latest_value_for_year(
        facts,
        tags=["NetCashProvidedByUsedInOperatingActivities"],
        fiscal_year=year,
    ),
    capex=_extract_latest_value_for_year(
        facts,
        tags=["CapitalExpenditures", "PaymentsToAcquirePropertyPlantAndEquipment"],
        fiscal_year=year,
    ),
    cash_and_equivalents=_extract_latest_value_for_year(
        facts,
        tags=["CashAndCashEquivalentsAtCarryingValue"],
        fiscal_year=year,
    ),
    total_debt=_build_total_debt(facts, fiscal_year=year),
    total_equity=_extract_latest_value_for_year(
        facts,
        tags=["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
        fiscal_year=year,
    ),
)

## Why the Schema Layer Matters

The schema layer mattered because it changed the workflow from raw-source handling into reusable analytical processing.

Once retrieved records were converted into annual snapshots, downstream functions could operate on a stable internal format instead of repeatedly interpreting source-specific responses. This reduced repeated complexity and improved separation of concerns across the pipeline.

From a technical perspective, the schema layer created three important benefits.

First, it improved maintainability by isolating retrieval-specific logic from downstream transformations.  
Second, it improved extensibility by making it easier to add new metrics or output views without redesigning the ingestion layer.  
Third, it improved interpretability by making each annual record explicit, reviewable, and comparison-ready.

For that reason, the schema layer should be understood as a structural foundation of the workflow rather than as a simple implementation convenience.

In [ ]:
def _build_snapshot_df(snapshots):
    rows = []
    for s in snapshots:
        rows.append({
            "Year": getattr(s, "fiscal_year", None),
            "Revenue": getattr(s, "revenue", None),
            "Net Income": getattr(s, "net_income", None),
            "Operating Income": getattr(s, "operating_income", None),
            "Operating Cash Flow": getattr(s, "operating_cash_flow", None),
            "CapEx": getattr(s, "capex", None),
            "Cash": getattr(s, "cash_and_equivalents", None),
            "Total Debt": getattr(s, "total_debt", None),
            "Total Equity": getattr(s, "total_equity", None),
            "Current Assets": getattr(s, "current_assets", None),
            "Current Liabilities": getattr(s, "current_liabilities", None),
            "Diluted Shares": getattr(s, "shares_outstanding_diluted", None),
            "Basic Shares": getattr(s, "shares_outstanding_basic", None),
        })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    if "Year" in df.columns:
        df = df.sort_values("Year", na_position="last")

    return df

## Validation Perspective

The schema layer improved consistency, but it did not eliminate every data risk.

A standardized internal structure can reduce repeated complexity, but it cannot guarantee that source values are complete, perfectly aligned by fiscal year, or equally comparable across all cases. If a retrieved field is incomplete, inconsistently tagged, or semantically ambiguous, the internal snapshot may preserve that limitation rather than resolve it.

This distinction was important in the project because it clarified the boundary between structural standardization and analytical certainty. The schema layer made downstream analysis more feasible and more maintainable, but validation still remained necessary before strong interpretation could be made.

## Transition

Once the internal data model was established, the next step was to transform those annual records into reusable analytical variables.

The following chapter explains how the project engineered growth, profitability, cash flow quality, financial condition, and valuation-related metrics from the standardized snapshot layer.